# 02. AlphaZero Implementation (Track A)

## CS377 Reinforcement Learning — Team 1
### Concentration vs. Dispersion in Free-Setup Chess via AlphaZero

---

This notebook contains the **complete AlphaZero implementation** for Track A:

1. **Game ABC** — Game interface abstraction
2. **TicTacToe** — Verification target (simple game)
3. **Chess Game** — Standard chess with handicap FEN support
4. **Board & Action Encoding** — AlphaZero 21-plane / 8×8×73 scheme
5. **Neural Network** — Dual-headed ResNet (policy + value)
6. **MCTS** — PUCT Monte Carlo Tree Search
7. **Self-Play** — Batched self-play with multi-GPU support
8. **Trainer** — Training loop (policy CE + value MSE + L2)
9. **Arena & Baselines** — Evaluation framework
10. **Training Script** — CLI entry point

---
## 1. Game Abstract Base Class

게임-무관한 AlphaZero 코어(MCTS, 신경망, self-play, 학습)와  
게임-특화 로직을 분리하는 인터페이스.

Source: `handichess/track_a/game/base.py`

In [ ]:
"""
Game Abstract Base Class.

Defines the interface that separates the game-agnostic AlphaZero core
(MCTS, neural network, self-play, training) from game-specific logic.
Implementations: TicTacToe (for verification), standard chess (with handicap).
"""

from __future__ import annotations

from abc import ABC, abstractmethod
from typing import Optional

import numpy as np


class Game(ABC):
    """
    Abstract base class for a two-player zero-sum game.

    Convention:
      - Player 1 = +1, Player 2 = -1.
      - Board state is a numpy array.
      - Canonical form: the board as seen from the current player's perspective.
    """

    @abstractmethod
    def get_init_board(self) -> np.ndarray:
        """Return the initial board state."""
        ...

    @abstractmethod
    def get_board_size(self) -> tuple[int, ...]:
        """Return the board dimensions, e.g. (3,3) for TicTacToe, (8,8) for chess."""
        ...

    @abstractmethod
    def get_action_size(self) -> int:
        """Return the total number of possible actions (including illegal ones)."""
        ...

    @abstractmethod
    def get_next_state(
        self, board: np.ndarray, player: int, action: int
    ) -> tuple[np.ndarray, int]:
        """Apply an action and return the new board and the next player."""
        ...

    @abstractmethod
    def get_valid_moves(self, board: np.ndarray, player: int) -> np.ndarray:
        """Return a binary mask of valid actions."""
        ...

    @abstractmethod
    def get_game_ended(self, board: np.ndarray, player: int) -> float:
        """
        Check if the game has ended.
        Returns: 0 if not over, +1 if player won, -1 if player lost, 1e-4 for draw.
        """
        ...

    @abstractmethod
    def get_canonical_form(self, board: np.ndarray, player: int) -> np.ndarray:
        """Return the board from the perspective of the given player."""
        ...

    @abstractmethod
    def get_encoded_state(self, board: np.ndarray) -> np.ndarray:
        """Encode the board as input planes for the neural network."""
        ...

    @abstractmethod
    def string_representation(self, board: np.ndarray) -> str:
        """Return a unique string representation of the board state."""
        ...

    def get_symmetries(
        self, board: np.ndarray, pi: np.ndarray
    ) -> list[tuple[np.ndarray, np.ndarray]]:
        """Return (board, policy) pairs representing symmetries. Default: no symmetries."""
        return [(board, pi)]

    def display(self, board: np.ndarray) -> None:
        """Print a human-readable board. Optional."""
        print(self.string_representation(board))

---
## 2. TicTacToe (Verification Target)

AlphaZero 코어 검증용 간단한 게임. 학습 완료 시 절대 지지 않는 완벽한 플레이가 가능해야 합니다.

Source: `handichess/track_a/game/tictactoe.py`

In [ ]:
"""
TicTacToe implementation of the Game ABC.

Used as a verification target: the AlphaZero core should learn perfect
play (never lose) on this simple game before moving to chess.
"""

from __future__ import annotations

import numpy as np


class TicTacToeGame(Game):
    """
    3×3 TicTacToe.

    Board representation: 3×3 numpy array.
      0 = empty, +1 = player 1 (X), -1 = player 2 (O).

    Action space: 9 positions (row * 3 + col).
    """

    def __init__(self):
        self._board_size = (3, 3)
        self._action_size = 9

    def get_init_board(self) -> np.ndarray:
        return np.zeros(self._board_size, dtype=np.float32)

    def get_board_size(self) -> tuple[int, int]:
        return self._board_size

    def get_action_size(self) -> int:
        return self._action_size

    def get_next_state(
        self, board: np.ndarray, player: int, action: int
    ) -> tuple[np.ndarray, int]:
        b = board.copy()
        row, col = divmod(action, 3)
        assert b[row, col] == 0, f"Illegal move: position ({row},{col}) is occupied"
        b[row, col] = player
        return b, -player

    def get_valid_moves(self, board: np.ndarray, player: int) -> np.ndarray:
        valid = np.zeros(self._action_size, dtype=np.float32)
        for i in range(9):
            row, col = divmod(i, 3)
            if board[row, col] == 0:
                valid[i] = 1.0
        return valid

    def get_game_ended(self, board: np.ndarray, player: int) -> float:
        for i in range(3):
            if abs(board[i].sum()) == 3:
                return 1.0 if board[i, 0] == player else -1.0
            if abs(board[:, i].sum()) == 3:
                return 1.0 if board[0, i] == player else -1.0

        diag1 = board[0, 0] + board[1, 1] + board[2, 2]
        if abs(diag1) == 3:
            return 1.0 if board[1, 1] == player else -1.0

        diag2 = board[0, 2] + board[1, 1] + board[2, 0]
        if abs(diag2) == 3:
            return 1.0 if board[1, 1] == player else -1.0

        if not np.any(board == 0):
            return 1e-4

        return 0.0

    def get_canonical_form(self, board: np.ndarray, player: int) -> np.ndarray:
        return board * player

    def get_encoded_state(self, board: np.ndarray) -> np.ndarray:
        """
        Encode as 3 planes:
          plane 0: current player's pieces (1 where board > 0)
          plane 1: opponent's pieces (1 where board < 0)
          plane 2: empty squares (1 where board == 0)
        """
        encoded = np.zeros((3, 3, 3), dtype=np.float32)
        encoded[0] = (board > 0).astype(np.float32)
        encoded[1] = (board < 0).astype(np.float32)
        encoded[2] = (board == 0).astype(np.float32)
        return encoded

    def string_representation(self, board: np.ndarray) -> str:
        return board.tobytes().hex()

    def get_symmetries(
        self, board: np.ndarray, pi: np.ndarray
    ) -> list[tuple[np.ndarray, np.ndarray]]:
        """TicTacToe has 8 symmetries (4 rotations × 2 reflections)."""
        assert len(pi) == self._action_size
        pi_board = pi.reshape(3, 3)
        symmetries = []

        for rot in range(4):
            b_rot = np.rot90(board, rot)
            p_rot = np.rot90(pi_board, rot)
            symmetries.append((b_rot.copy(), p_rot.flatten().copy()))
            b_flip = np.fliplr(b_rot)
            p_flip = np.fliplr(p_rot)
            symmetries.append((b_flip.copy(), p_flip.flatten().copy()))

        return symmetries

    def display(self, board: np.ndarray) -> None:
        symbols = {1: "X", -1: "O", 0: "."}
        print("  0 1 2")
        for i in range(3):
            row_str = " ".join(symbols[int(board[i, j])] for j in range(3))
            print(f"{i} {row_str}")
        print()

---
## 3. Board & Action Encoding (AlphaZero Scheme)

체스 보드를 신경망 입력으로 인코딩합니다:
- **Board**: 21 input planes (12 piece + 2 repetition + 1 color + 1 move count + 4 castling + 1 fifty-move)
- **Action**: 8×8×73 = 4672 (56 queen moves + 8 knight moves + 9 underpromotions)

Source: `handichess/track_a/encoding.py`

In [ ]:
"""
Board and action encoding for chess, following the AlphaZero scheme.

Board encoding:
  - 12 piece planes (P, N, B, R, Q, K for each color)
  - 2 repetition planes (1-fold, 2-fold repetition)
  - 1 color plane (all 1s if white to move, all 0s if black)
  - 1 total move count (normalized)
  - 4 castling rights (K, Q, k, q)
  - 1 no-progress count (normalized fifty-move counter)
  Total: 21 input planes of 8×8

Action encoding (AlphaZero 8×8×73):
  - 56 "queen moves" (7 distances × 8 directions)
  - 8 knight moves
  - 9 underpromotions (3 piece types × 3 directions)
  Total: 73 planes, from-square determines the 8×8 position.

All encoding is from the perspective of the current player (canonical form).
"""

from __future__ import annotations

import chess
import numpy as np
from typing import Optional


# ── Constants ────────────────────────────────────────────────────────────────
NUM_INPUT_PLANES = 21
ACTION_PLANES = 73
ACTION_SIZE = 8 * 8 * ACTION_PLANES  # 4672

# Piece type to plane index (current player's pieces first)
_PIECE_PLANES = {
    (chess.PAWN, True): 0,
    (chess.KNIGHT, True): 1,
    (chess.BISHOP, True): 2,
    (chess.ROOK, True): 3,
    (chess.QUEEN, True): 4,
    (chess.KING, True): 5,
    (chess.PAWN, False): 6,
    (chess.KNIGHT, False): 7,
    (chess.BISHOP, False): 8,
    (chess.ROOK, False): 9,
    (chess.QUEEN, False): 10,
    (chess.KING, False): 11,
}

# Queen-move direction vectors (file_delta, rank_delta)
_QUEEN_DIRECTIONS = [
    (0, 1), (1, 1), (1, 0), (1, -1),
    (0, -1), (-1, -1), (-1, 0), (-1, 1),
]

_KNIGHT_MOVES = [
    (1, 2), (2, 1), (2, -1), (1, -2),
    (-1, -2), (-2, -1), (-2, 1), (-1, 2),
]

_UNDERPROMOTION_PIECES = [chess.ROOK, chess.BISHOP, chess.KNIGHT]
_UNDERPROMOTION_DIRS = [(-1, 1), (0, 1), (1, 1)]


# ── Board encoding ──────────────────────────────────────────────────────────
def encode_board(board: chess.Board) -> np.ndarray:
    """Encode a chess board as a (21, 8, 8) float32 tensor."""
    planes = np.zeros((NUM_INPUT_PLANES, 8, 8), dtype=np.float32)
    turn = board.turn

    # Piece planes (0-11)
    for sq in chess.SQUARES:
        piece = board.piece_at(sq)
        if piece is not None:
            is_own = piece.color == turn
            plane_idx = _PIECE_PLANES[(piece.piece_type, is_own)]
            rank = chess.square_rank(sq)
            file = chess.square_file(sq)
            planes[plane_idx, rank, file] = 1.0

    # Repetition planes (12-13)
    if board.is_repetition(1):
        planes[12, :, :] = 1.0
    if board.is_repetition(2):
        planes[13, :, :] = 1.0

    # Color plane (14)
    if turn == chess.WHITE:
        planes[14, :, :] = 1.0

    # Move count (15)
    planes[15, :, :] = min(board.fullmove_number / 200.0, 1.0)

    # Castling rights (16-19)
    if board.has_kingside_castling_rights(chess.WHITE):
        planes[16, :, :] = 1.0
    if board.has_queenside_castling_rights(chess.WHITE):
        planes[17, :, :] = 1.0
    if board.has_kingside_castling_rights(chess.BLACK):
        planes[18, :, :] = 1.0
    if board.has_queenside_castling_rights(chess.BLACK):
        planes[19, :, :] = 1.0

    # No-progress count (20)
    planes[20, :, :] = board.halfmove_clock / 100.0

    return planes


# ── Action encoding ──────────────────────────────────────────────────────────
def encode_action(move: chess.Move, board: chess.Board) -> int:
    """Encode a chess move into an action index in [0, 4672)."""
    from_sq = move.from_square
    to_sq = move.to_square
    from_rank = chess.square_rank(from_sq)
    from_file = chess.square_file(from_sq)
    to_rank = chess.square_rank(to_sq)
    to_file = chess.square_file(to_sq)

    d_file = to_file - from_file
    d_rank = to_rank - from_rank

    plane = _get_move_plane(d_file, d_rank, move.promotion, board.piece_type_at(from_sq))

    return (from_rank * 8 + from_file) * ACTION_PLANES + plane


def decode_action(action: int, board: chess.Board) -> chess.Move:
    """Decode an action index back to a chess move."""
    sq_idx = action // ACTION_PLANES
    plane = action % ACTION_PLANES
    from_rank = sq_idx // 8
    from_file = sq_idx % 8
    from_sq = chess.square(from_file, from_rank)

    d_file, d_rank, promotion = _decode_move_plane(plane)

    piece = board.piece_at(from_sq)
    if piece is not None and piece.color == chess.BLACK and plane >= 64:
        d_rank = -d_rank

    to_file = from_file + d_file
    to_rank = from_rank + d_rank
    to_sq = chess.square(to_file, to_rank)

    piece = board.piece_type_at(from_sq)
    if piece == chess.PAWN and promotion is None:
        if (board.turn == chess.WHITE and to_rank == 7) or \
           (board.turn == chess.BLACK and to_rank == 0):
            promotion = chess.QUEEN

    return chess.Move(from_sq, to_sq, promotion=promotion)


def _get_move_plane(
    d_file: int, d_rank: int,
    promotion: Optional[int],
    piece_type: Optional[int],
) -> int:
    """
    Determine the move plane index (0-72) from move deltas and context.
    Planes 0-55:  Queen moves (7 distances × 8 directions)
    Planes 56-63: Knight moves (8 moves)
    Planes 64-72: Underpromotions (3 pieces × 3 directions)
    """
    # Underpromotion
    if promotion is not None and promotion != chess.QUEEN:
        piece_idx = _UNDERPROMOTION_PIECES.index(promotion)
        for dir_idx, (df, dr) in enumerate(_UNDERPROMOTION_DIRS):
            if d_file == df and d_rank == dr:
                return 64 + piece_idx * 3 + dir_idx
        for dir_idx, (df, dr) in enumerate(_UNDERPROMOTION_DIRS):
            if d_file == df and d_rank == -dr:
                return 64 + piece_idx * 3 + dir_idx
        raise ValueError(f"Cannot encode underpromotion: df={d_file}, dr={d_rank}, prom={promotion}")

    # Knight move
    if piece_type == chess.KNIGHT or (d_file, d_rank) in _KNIGHT_MOVES:
        try:
            knight_idx = _KNIGHT_MOVES.index((d_file, d_rank))
            return 56 + knight_idx
        except ValueError:
            pass

    # Queen move
    distance = max(abs(d_file), abs(d_rank))
    assert distance >= 1, f"Zero-distance move: df={d_file}, dr={d_rank}"

    unit_f = (d_file // distance) if d_file != 0 else 0
    unit_r = (d_rank // distance) if d_rank != 0 else 0

    try:
        dir_idx = _QUEEN_DIRECTIONS.index((unit_f, unit_r))
    except ValueError:
        raise ValueError(
            f"Cannot find direction for ({unit_f}, {unit_r}) "
            f"from delta ({d_file}, {d_rank})"
        )

    return dir_idx * 7 + (distance - 1)


def _decode_move_plane(plane: int) -> tuple[int, int, Optional[int]]:
    """Decode a move plane index to (d_file, d_rank, promotion)."""
    if plane < 56:
        dir_idx = plane // 7
        distance = (plane % 7) + 1
        d_file, d_rank = _QUEEN_DIRECTIONS[dir_idx]
        return d_file * distance, d_rank * distance, None
    elif plane < 64:
        knight_idx = plane - 56
        d_file, d_rank = _KNIGHT_MOVES[knight_idx]
        return d_file, d_rank, None
    else:
        idx = plane - 64
        piece_idx = idx // 3
        dir_idx = idx % 3
        d_file, d_rank = _UNDERPROMOTION_DIRS[dir_idx]
        promotion = _UNDERPROMOTION_PIECES[piece_idx]
        return d_file, d_rank, promotion


# ── Legal move mask ──────────────────────────────────────────────────────────
def get_legal_move_mask(board: chess.Board) -> np.ndarray:
    """Generate a binary mask for all legal moves."""
    mask = np.zeros(ACTION_SIZE, dtype=np.float32)
    for move in board.legal_moves:
        try:
            action = encode_action(move, board)
            mask[action] = 1.0
        except (ValueError, AssertionError):
            pass
    return mask


def encode_action_batch(
    moves: list[chess.Move], board: chess.Board
) -> list[int]:
    """Encode a batch of moves."""
    return [encode_action(m, board) for m in moves]

---
## 4. Chess Game Implementation

python-chess를 래핑한 체스 구현. 핸디캡 FEN에서 시작하는 게임을 지원합니다.  
상태 표현: `START_FEN\nMOVE1 MOVE2 ...` (전체 무브 히스토리 보존).

Source: `handichess/track_a/game/chess_std.py`

In [ ]:
"""
Standard Chess implementation of the Game ABC.

Wraps python-chess for full standard rules (castling, en passant, promotion,
all draw rules). The only customization is the initial position: games can
start from a handicap FEN instead of the standard position.

Action encoding follows AlphaZero's 8×8×73 scheme.
"""

from __future__ import annotations

from typing import Optional

import chess
import numpy as np


class ChessGame(Game):
    """
    Standard chess with optional handicap starting position.
    State: numpy array storing FEN + full move history.
    """

    _MAX_STATE_LEN = 4096

    def __init__(self, start_fen: Optional[str] = None, max_moves: int = 180):
        self.start_fen = start_fen or chess.STARTING_FEN
        self.max_moves = max_moves
        self._board_size = (8, 8)
        self._action_size = 8 * 8 * ACTION_PLANES  # 4672
        self._state_cache: dict[str, chess.Board] = {}

    def get_init_board(self) -> np.ndarray:
        return self._encode_state(self.start_fen, [])

    def get_board_size(self) -> tuple[int, int]:
        return self._board_size

    def get_action_size(self) -> int:
        return self._action_size

    def get_next_state(self, board: np.ndarray, player: int, action: int) -> tuple[np.ndarray, int]:
        b = self._state_to_board(board)
        move = decode_action(action, b)
        assert move in b.legal_moves, f"Illegal move: {move} in position {b.fen()}"
        
        start_fen, moves = self._decode_state(board)
        moves.append(move.uci())
        new_state = self._encode_state(start_fen, moves)
        
        new_key = bytes(new_state[new_state > 0].tolist()).decode("ascii")
        b_next = b.copy()
        b_next.push(move)
        
        if len(self._state_cache) > 100000:
            self._state_cache.clear()
        self._state_cache[new_key] = b_next
        
        return new_state, -player

    def get_valid_moves(self, board: np.ndarray, player: int) -> np.ndarray:
        b = self._state_to_board(board)
        return get_legal_move_mask(b)

    def get_game_ended(self, board: np.ndarray, player: int) -> float:
        b = self._state_to_board(board)

        if b.fullmove_number * 2 - (1 if b.turn == chess.WHITE else 0) >= self.max_moves:
            return 1e-4

        outcome = b.outcome(claim_draw=True)
        if outcome is None:
            return 0.0

        if outcome.winner is None:
            return 1e-4

        winner_is_white = outcome.winner == chess.WHITE
        if (winner_is_white and player == 1) or (not winner_is_white and player == -1):
            return 1.0
        else:
            return -1.0

    def get_canonical_form(self, board: np.ndarray, player: int) -> np.ndarray:
        """
        No-op canonical form for chess.
        encode_board() already handles perspective correctly via is_own check.
        """
        return board.copy()

    def get_encoded_state(self, board: np.ndarray) -> np.ndarray:
        b = self._state_to_board(board)
        return encode_board(b)

    def string_representation(self, board: np.ndarray) -> str:
        b = self._state_to_board(board)
        return b.fen()

    def display(self, board: np.ndarray) -> None:
        b = self._state_to_board(board)
        print(b)
        print(f"FEN: {b.fen()}")
        print()

    # ── Internal helpers ────────────────────────────────────────────

    def _encode_state(self, start_fen: str, moves: list[str]) -> np.ndarray:
        arr = np.zeros(self._MAX_STATE_LEN, dtype=np.uint8)
        text = start_fen
        if moves:
            text += "\n" + " ".join(moves)
        text_bytes = text.encode("ascii")
        arr[: len(text_bytes)] = list(text_bytes)
        return arr

    def _decode_state(self, arr: np.ndarray) -> tuple[str, list[str]]:
        text = bytes(arr[arr > 0].tolist()).decode("ascii")
        parts = text.split("\n", 1)
        start_fen = parts[0]
        if len(parts) > 1 and parts[1].strip():
            moves = parts[1].strip().split()
        else:
            moves = []
        return start_fen, moves

    def _replay_board(self, start_fen: str, moves: list[str]) -> chess.Board:
        board = chess.Board(start_fen)
        for uci in moves:
            board.push(chess.Move.from_uci(uci))
        return board

    def _state_to_board(self, arr: np.ndarray) -> chess.Board:
        key = bytes(arr[arr > 0].tolist()).decode("ascii")
        if key in self._state_cache:
            return self._state_cache[key]
        start_fen, moves = self._decode_state(arr)
        board = self._replay_board(start_fen, moves)
        if len(self._state_cache) > 100000:
            self._state_cache.clear()
        self._state_cache[key] = board
        return board

    @classmethod
    def from_matchup(cls, pattern_id: str, noq_color: str = "white", max_moves: int = 180) -> "ChessGame":
        """Create a ChessGame with a match-up starting position."""
        side = chess.WHITE if noq_color == "white" else chess.BLACK
        pattern = get_pattern_by_id(pattern_id)
        pos = generate_position(pattern, side)
        return cls(start_fen=pos.fen, max_moves=max_moves)

---
## 5. Neural Network (Dual-Headed ResNet)

AlphaZero 아키텍처:
- **Shared backbone**: 10 residual blocks (128 channels)
- **Policy head**: action probability distribution (softmax)
- **Value head**: scalar evaluation (tanh, [-1, +1])

Source: `handichess/track_a/net.py`

In [ ]:
"""
Residual neural network for AlphaZero.

Dual-headed architecture:
  - Shared ResNet backbone (configurable depth and width)
  - Policy head → action probabilities (softmax over action space)
  - Value head → scalar evaluation (tanh, range [-1, +1])
"""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional


class ResidualBlock(nn.Module):
    """A single residual block: conv → BN → ReLU → conv → BN + skip → ReLU."""

    def __init__(self, num_channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(num_channels, num_channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.conv2 = nn.Conv2d(num_channels, num_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(num_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + residual
        return F.relu(out)


class AlphaZeroNet(nn.Module):
    """
    AlphaZero-style dual-headed residual network.

    Input:  (batch, num_input_planes, board_h, board_w)
    Output: (policy_logits, value)
            policy_logits: (batch, action_size)
            value: (batch, 1) — scalar in [-1, +1]
    """

    def __init__(self, num_input_planes: int, board_size: tuple[int, int],
                 action_size: int, num_res_blocks: int = 10,
                 num_channels: int = 128, dropout: float = 0.3):
        super().__init__()
        self.board_h, self.board_w = board_size
        self.action_size = action_size

        self.conv_input = nn.Conv2d(num_input_planes, num_channels, 3, padding=1, bias=False)
        self.bn_input = nn.BatchNorm2d(num_channels)

        self.res_blocks = nn.ModuleList(
            [ResidualBlock(num_channels) for _ in range(num_res_blocks)]
        )

        # Policy head
        self.policy_conv = nn.Conv2d(num_channels, 32, 1, bias=False)
        self.policy_bn = nn.BatchNorm2d(32)
        self.policy_fc = nn.Linear(32 * self.board_h * self.board_w, action_size)

        # Value head
        self.value_conv = nn.Conv2d(num_channels, 1, 1, bias=False)
        self.value_bn = nn.BatchNorm2d(1)
        self.value_fc1 = nn.Linear(self.board_h * self.board_w, 256)
        self.value_dropout = nn.Dropout(dropout)
        self.value_fc2 = nn.Linear(256, 1)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        out = F.relu(self.bn_input(self.conv_input(x)))
        for block in self.res_blocks:
            out = block(out)

        # Policy head
        p = F.relu(self.policy_bn(self.policy_conv(out)))
        p = p.view(p.size(0), -1)
        p = self.policy_fc(p)

        # Value head
        v = F.relu(self.value_bn(self.value_conv(out)))
        v = v.view(v.size(0), -1)
        v = F.relu(self.value_fc1(v))
        v = torch.tanh(self.value_fc2(v))

        return p, v

    def predict(self, encoded_state: torch.Tensor, valid_moves: Optional[torch.Tensor] = None):
        """Convenience method for single-state or batched prediction."""
        self.eval()
        with torch.no_grad():
            is_batched = encoded_state.dim() == 4
            x = encoded_state if is_batched else encoded_state.unsqueeze(0)
            policy_logits, value = self(x)

            if valid_moves is not None:
                v_mask = valid_moves if is_batched else valid_moves.unsqueeze(0)
                policy_logits = policy_logits.masked_fill(v_mask == 0, float("-inf"))

            policy_probs = F.softmax(policy_logits, dim=1)
            if is_batched:
                return policy_probs, value
            else:
                return policy_probs[0], value[0]


def create_net_for_game(game, config: Optional[dict] = None) -> AlphaZeroNet:
    """Factory function to create a network for a specific game."""
    if config is None:
        config = {}
    board = game.get_init_board()
    canonical = game.get_canonical_form(board, 1)
    encoded = game.get_encoded_state(canonical)
    num_input_planes = encoded.shape[0]

    return AlphaZeroNet(
        num_input_planes=num_input_planes,
        board_size=game.get_board_size(),
        action_size=game.get_action_size(),
        num_res_blocks=config.get("num_res_blocks", 10),
        num_channels=config.get("num_channels", 128),
        dropout=config.get("dropout", 0.3),
    )

---
## 6. Monte Carlo Tree Search (MCTS)

PUCT 알고리즘 기반의 MCTS 구현.  
Batched leaf evaluation을 위한 `find_leaf` / `expand_and_backup` 분리 지원.

Source: `handichess/track_a/mcts.py`

In [ ]:
"""
Monte Carlo Tree Search (MCTS) with PUCT exploration.

Supports both unbatched search() for single evaluations and step-based
methods (find_leaf, expand_and_backup) for Batched Game Self-Play.
"""

from __future__ import annotations

import math
from typing import Optional

import numpy as np
import torch


class MCTSNode:
    """A node in the MCTS tree."""

    __slots__ = [
        "game", "state", "player", "parent", "action",
        "children", "visit_count", "value_sum", "prior",
        "is_expanded", "is_terminal", "terminal_value",
    ]

    def __init__(self, game, state, player, parent=None, action=None, prior=0.0):
        self.game = game
        self.state = state
        self.player = player
        self.parent = parent
        self.action = action
        self.prior = prior

        self.children: dict[int, MCTSNode] = {}
        self.visit_count = 0
        self.value_sum = 0.0
        self.is_expanded = False
        self.is_terminal = False
        self.terminal_value = 0.0

    @property
    def q_value(self) -> float:
        if self.visit_count == 0:
            return 0.0
        return self.value_sum / self.visit_count


class MCTS:
    """AlphaZero-style MCTS."""

    def __init__(self, game, net, num_simulations=800, c_puct=1.25,
                 dirichlet_alpha=0.3, dirichlet_epsilon=0.25, device="cpu"):
        self.game = game
        self.net = net
        self.num_simulations = num_simulations
        self.c_puct = c_puct
        self.dirichlet_alpha = dirichlet_alpha
        self.dirichlet_epsilon = dirichlet_epsilon
        self.device = device

    def search(self, state, player, temperature=1.0, add_noise=True) -> np.ndarray:
        """Unbatched search (for Arena / single evaluations)."""
        root = MCTSNode(self.game, state, player)
        
        # Initial expansion
        leaf_data = self.find_leaf(root)
        if leaf_data is not None:
            search_path, node, encoded, valid = leaf_data
            enc_t = torch.FloatTensor(encoded).unsqueeze(0).to(self.device)
            val_t = torch.FloatTensor(valid).unsqueeze(0).to(self.device)
            p, v = self.net.predict(enc_t, val_t)
            self.expand_and_backup(search_path, node, p[0].cpu().numpy(), v[0].item(), valid)

        if add_noise:
            self.add_dirichlet_noise(root)

        for _ in range(self.num_simulations):
            leaf_data = self.find_leaf(root)
            if leaf_data is not None:
                search_path, node, encoded, valid = leaf_data
                enc_t = torch.FloatTensor(encoded).unsqueeze(0).to(self.device)
                val_t = torch.FloatTensor(valid).unsqueeze(0).to(self.device)
                p, v = self.net.predict(enc_t, val_t)
                self.expand_and_backup(search_path, node, p[0].cpu().numpy(), v[0].item(), valid)

        return self.get_action_probs(root, temperature)

    def find_leaf(self, root):
        """Traverse to find unexpanded leaf. Returns None if terminal."""
        node = root
        search_path = [node]

        while node.is_expanded and not node.is_terminal:
            _, node = self._select_child(node)
            search_path.append(node)

        if node.state is None:
            parent = node.parent
            assert parent is not None and node.action is not None
            node.state, node.player = self.game.get_next_state(
                parent.state, parent.player, node.action
            )

        if node.is_terminal:
            self.backup(search_path, node.terminal_value)
            return None

        game_result = self.game.get_game_ended(node.state, node.player)
        if game_result != 0:
            node.is_terminal = True
            node.terminal_value = game_result
            node.is_expanded = True
            self.backup(search_path, game_result)
            return None

        canonical = self.game.get_canonical_form(node.state, node.player)
        encoded = self.game.get_encoded_state(canonical)
        valid_moves = self.game.get_valid_moves(node.state, node.player)

        return search_path, node, encoded, valid_moves

    def expand_and_backup(self, search_path, node, policy_probs, value, valid_moves=None):
        """Create children from neural net policy and backpropagate value."""
        if valid_moves is None:
            valid_moves = self.game.get_valid_moves(node.state, node.player)

        legal_actions = np.nonzero(valid_moves)[0]
        for action in legal_actions:
            child = MCTSNode(
                game=self.game, state=None, player=-node.player,
                parent=node, action=action, prior=policy_probs[action],
            )
            node.children[action] = child

        node.is_expanded = True
        self.backup(search_path, value)

    def _select_child(self, node):
        """Select child with highest PUCT score."""
        best_score = -float("inf")
        best_action = -1
        best_child = None

        sqrt_parent = math.sqrt(max(1, node.visit_count))

        for action, child in node.children.items():
            q = -child.q_value
            u = self.c_puct * child.prior * sqrt_parent / (1 + child.visit_count)
            score = q + u

            if score > best_score:
                best_score = score
                best_action = action
                best_child = child

        return best_action, best_child

    def backup(self, search_path, value):
        """Propagate value up the search path. Flips sign at each level."""
        for node in reversed(search_path):
            node.value_sum += value
            node.visit_count += 1
            value = -value

    def add_dirichlet_noise(self, root):
        """Adds exploration noise to root children priors."""
        if root.children:
            noise = np.random.dirichlet([self.dirichlet_alpha] * len(root.children))
            for i, child in enumerate(root.children.values()):
                child.prior = (
                    (1 - self.dirichlet_epsilon) * child.prior
                    + self.dirichlet_epsilon * noise[i]
                )

    def get_action_probs(self, root, temperature):
        """Convert root visit counts to action probabilities."""
        action_size = self.game.get_action_size()
        counts = np.zeros(action_size, dtype=np.float64)

        for action, child in root.children.items():
            counts[action] = child.visit_count

        if temperature == 0:
            best_actions = np.where(counts == counts.max())[0]
            probs = np.zeros(action_size, dtype=np.float64)
            probs[np.random.choice(best_actions)] = 1.0
            return probs.astype(np.float32)

        counts_temp = counts ** (1.0 / temperature)
        total = counts_temp.sum()
        if total == 0:
            valid = self.game.get_valid_moves(root.state, root.player)
            return (valid / valid.sum()).astype(np.float32)

        probs = counts_temp / total
        return probs.astype(np.float32)

---
## 7. Self-Play Pipeline

Batched MCTS 평가를 사용하여 수백 개의 게임을 동시에 실행하고,  
Multi-GPU 및 멀티프로세스 지원으로 GPU 활용률을 극대화합니다.

Source: `handichess/track_a/selfplay.py`

In [ ]:
"""
Self-play pipeline for AlphaZero training.

Generates training data by having the neural network play against itself.
Supports Batched MCTS evaluations to run hundreds of games concurrently
and saturate multi-GPU setups.
"""

from __future__ import annotations

import logging
import random
import time
from collections import deque
from dataclasses import dataclass
from typing import Optional

import numpy as np
import torch
import torch.multiprocessing as mp

logger = logging.getLogger(__name__)


@dataclass
class TrainingExample:
    """A single training example from self-play."""
    encoded_state: np.ndarray   # (planes, h, w) — canonical form
    policy_target: np.ndarray   # (action_size,) — MCTS visit probabilities
    value_target: float         # Game outcome: +1, 0, -1


class ReplayBuffer:
    """Fixed-size replay buffer for training examples."""

    def __init__(self, max_size: int = 100_000):
        self.buffer: deque[TrainingExample] = deque(maxlen=max_size)

    def add(self, examples: list[TrainingExample]) -> None:
        self.buffer.extend(examples)

    def sample(self, batch_size: int) -> list[TrainingExample]:
        return random.sample(list(self.buffer), min(batch_size, len(self.buffer)))

    def __len__(self) -> int:
        return len(self.buffer)

    def clear(self) -> None:
        self.buffer.clear()


class ActiveGame:
    """State wrapper for a concurrently running self-play game."""
    def __init__(self, game, mcts, start_state):
        self.game = game
        self.mcts = mcts
        self.state = start_state
        self.player = 1
        self.move_count = 0
        self.root = MCTSNode(game, start_state, 1)
        self.trajectory: list[tuple[np.ndarray, np.ndarray, int]] = []


class SelfPlay:
    """
    Batched Self-play game generator.
    Plays games concurrently to batch leaf evaluations together.
    """

    def __init__(self, game, net, mcts_config=None, selfplay_config=None, device="cpu"):
        self.game = game
        self.net = net
        self.device = device

        mc = mcts_config or {}
        self.num_simulations = mc.get("num_simulations", 800)
        self.c_puct = mc.get("c_puct", 1.25)
        self.dirichlet_alpha = mc.get("dirichlet_alpha", 0.3)
        self.dirichlet_epsilon = mc.get("dirichlet_epsilon", 0.25)

        sc = selfplay_config or {}
        self.max_moves = sc.get("max_moves", 512)
        self.exploration_plies = sc.get("exploration_plies", 16)

        if self.device == "cuda" and torch.cuda.device_count() > 1:
            self.model = torch.nn.DataParallel(self.net)
        else:
            self.model = self.net

    def _predict_batch(self, enc_batch, val_batch):
        """Runs batched inference, utilizing DataParallel if available."""
        self.model.eval()
        with torch.no_grad():
            policy_logits, value = self.model(enc_batch)
            policy_logits = policy_logits.masked_fill(val_batch == 0, float("-inf"))
            policy_probs = torch.nn.functional.softmax(policy_logits, dim=1)
            return policy_probs, value

    def generate_games(self, num_games, start_states=None) -> list[TrainingExample]:
        """Play N games concurrently and collect all training examples."""
        all_examples = []
        active_games = []

        for _ in range(num_games):
            if start_states:
                start = random.choice(start_states)
            else:
                start = self.game.get_init_board()

            mcts = MCTS(
                game=self.game, net=self.net,
                num_simulations=self.num_simulations, c_puct=self.c_puct,
                dirichlet_alpha=self.dirichlet_alpha,
                dirichlet_epsilon=self.dirichlet_epsilon, device=self.device,
            )
            active_games.append(ActiveGame(self.game, mcts, start))

        self.model.eval()
        move_step = 0
        total_gpu_time = 0.0
        total_cpu_time = 0.0
        total_gpu_calls = 0
        finished_games = 0
        selfplay_start = time.time()

        logger.info(f"  [selfplay] Starting {num_games} games | sims={self.num_simulations} | max_moves={self.max_moves} | device={self.device}")

        while active_games:
            move_step_start = time.time()

            # 1. Expand all roots
            t0 = time.time()
            batch_data = []
            for g in active_games:
                res = g.mcts.find_leaf(g.root)
                if res is not None:
                    batch_data.append((g, res))
            cpu_find_time = time.time() - t0

            if batch_data:
                t0 = time.time()
                enc_batch = torch.FloatTensor(np.array([item[1][2] for item in batch_data])).to(self.device)
                val_batch = torch.FloatTensor(np.array([item[1][3] for item in batch_data])).to(self.device)
                p_batch, v_batch = self._predict_batch(enc_batch, val_batch)
                p_batch = p_batch.cpu().numpy()
                v_batch = v_batch.cpu().numpy()
                gpu_time = time.time() - t0
                total_gpu_time += gpu_time
                total_gpu_calls += 1

                for i, (g, res) in enumerate(batch_data):
                    search_path, node, _, valid = res
                    g.mcts.expand_and_backup(search_path, node, p_batch[i], float(v_batch[i][0]), valid)

            # 2. Add Dirichlet Noise
            for g in active_games:
                g.mcts.add_dirichlet_noise(g.root)

            # 3. Run MCTS Simulations concurrently
            sim_cpu_time = 0.0
            sim_gpu_time = 0.0
            sim_gpu_calls = 0
            sim_total_batch = 0

            for _ in range(self.num_simulations):
                t0 = time.time()
                batch_data = []
                for g in active_games:
                    res = g.mcts.find_leaf(g.root)
                    if res is not None:
                        batch_data.append((g, res))
                sim_cpu_time += time.time() - t0

                if batch_data:
                    t0 = time.time()
                    enc_batch = torch.FloatTensor(np.array([item[1][2] for item in batch_data])).to(self.device)
                    val_batch = torch.FloatTensor(np.array([item[1][3] for item in batch_data])).to(self.device)
                    p_batch, v_batch = self._predict_batch(enc_batch, val_batch)
                    p_batch = p_batch.cpu().numpy()
                    v_batch = v_batch.cpu().numpy()
                    dt = time.time() - t0
                    sim_gpu_time += dt
                    sim_gpu_calls += 1
                    sim_total_batch += len(batch_data)

                    for i, (g, res) in enumerate(batch_data):
                        search_path, node, _, valid = res
                        g.mcts.expand_and_backup(search_path, node, p_batch[i], float(v_batch[i][0]), valid)

            total_gpu_time += sim_gpu_time
            total_cpu_time += sim_cpu_time
            total_gpu_calls += sim_gpu_calls

            # 4. Advance states and check for termination
            next_active = []
            for g in active_games:
                temperature = 1.0 if g.move_count < self.exploration_plies else 0.1
                action_probs = g.mcts.get_action_probs(g.root, temperature)

                canonical = self.game.get_canonical_form(g.state, g.player)
                encoded = self.game.get_encoded_state(canonical)
                symmetries = self.game.get_symmetries(canonical, action_probs)
                for sym_board, sym_pi in symmetries:
                    sym_encoded = self.game.get_encoded_state(sym_board)
                    g.trajectory.append((sym_encoded, sym_pi, g.player))

                action = np.random.choice(len(action_probs), p=action_probs)
                g.state, g.player = self.game.get_next_state(g.state, g.player, action)
                g.move_count += 1

                result = self.game.get_game_ended(g.state, g.player)
                if result == 0 and g.move_count >= self.max_moves:
                    result = 1e-4

                if result != 0:
                    finished_games += 1
                    for encoded, policy, p in g.trajectory:
                        if abs(result) < 0.01:
                            value = 0.0
                        elif p == g.player:
                            value = result
                        else:
                            value = -result
                        all_examples.append(TrainingExample(
                            encoded_state=encoded, policy_target=policy, value_target=value,
                        ))
                else:
                    g.root = MCTSNode(self.game, g.state, g.player)
                    next_active.append(g)

            active_games = next_active
            move_step += 1

        total_elapsed = time.time() - selfplay_start
        logger.info(f"  [selfplay] DONE | {finished_games} games in {total_elapsed:.1f}s | examples={len(all_examples)}")
        return all_examples


def create_matchup_start_states(game_class, pattern_ids, max_moves=180):
    """Create a list of starting states from match-up positions."""
    states = []
    for pid in pattern_ids:
        for noq_color in ["white", "black"]:
            g = game_class.from_matchup(pid, noq_color, max_moves)
            states.append(g.get_init_board())
    return states


# ── MultiProcessSelfPlay omitted for brevity (see source for multi-GPU spawning) ──

---
## 8. Trainer (Training Loop)

Loss = Policy Cross-Entropy + Value MSE + L2 Regularization  
SGD with momentum 또는 Adam, step LR decay, AMP 지원.

Source: `handichess/track_a/trainer.py`

In [ ]:
"""
Training loop for AlphaZero.
Loss = policy_CE + value_MSE + L2_regularization.
Supports SGD with momentum or Adam, step LR decay, and AMP.
"""

from __future__ import annotations

import os
import logging
from pathlib import Path
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, TensorDataset

logger = logging.getLogger(__name__)


class Trainer:
    """
    AlphaZero trainer.
    Manages the training loop: takes training examples from self-play,
    trains the network, saves checkpoints.
    """

    def __init__(self, net, config=None, device="cpu", checkpoint_dir="runs/checkpoints"):
        self.net = net
        self.device = device
        self.net.to(device)

        if self.device == "cuda" and torch.cuda.device_count() > 1:
            self.model = nn.DataParallel(self.net)
        else:
            self.model = self.net

        c = config or {}
        self.epochs = c.get("epochs_per_iteration", 10)
        self.batch_size = c.get("batch_size", 256)
        self.lr = c.get("learning_rate", 0.01)
        self.weight_decay = c.get("weight_decay", 1e-4)
        self.momentum = c.get("momentum", 0.9)
        self.optimizer_type = c.get("optimizer", "sgd")
        self.use_amp = c.get("use_amp", False) and device != "cpu"
        self.lr_schedule = c.get("lr_schedule", [])
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

        if self.optimizer_type == "adam":
            self.optimizer = optim.Adam(net.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        else:
            self.optimizer = optim.SGD(net.parameters(), lr=self.lr, momentum=self.momentum, weight_decay=self.weight_decay)

        self.scaler = GradScaler("cuda" if self.device == "cuda" else "cpu", enabled=self.use_amp)
        self.value_loss_fn = nn.MSELoss()
        self.iteration = 0
        self.train_history: list[dict] = []

    def train(self, examples, iteration=None) -> dict:
        """Train the network on a batch of training examples."""
        if iteration is not None:
            self.iteration = iteration
            self._update_lr()

        states = np.array([e.encoded_state for e in examples])
        policies = np.array([e.policy_target for e in examples])
        values = np.array([e.value_target for e in examples], dtype=np.float32)

        states_t = torch.FloatTensor(states).to(self.device)
        policies_t = torch.FloatTensor(policies).to(self.device)
        values_t = torch.FloatTensor(values).to(self.device).unsqueeze(1)

        dataset = TensorDataset(states_t, policies_t, values_t)
        dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True, drop_last=False)

        self.model.train()
        total_loss = total_pi_loss = total_v_loss = 0.0
        num_batches = 0

        for epoch in range(self.epochs):
            for batch_states, batch_policies, batch_values in dataloader:
                self.optimizer.zero_grad()

                with autocast(device_type="cuda" if self.device == "cuda" else "cpu", enabled=self.use_amp):
                    policy_logits, value_pred = self.model(batch_states)
                    pi_loss = -torch.mean(torch.sum(batch_policies * torch.log_softmax(policy_logits, dim=1), dim=1))
                    v_loss = self.value_loss_fn(value_pred, batch_values)
                    loss = pi_loss + v_loss

                self.scaler.scale(loss).backward()
                self.scaler.step(self.optimizer)
                self.scaler.update()

                total_loss += loss.item()
                total_pi_loss += pi_loss.item()
                total_v_loss += v_loss.item()
                num_batches += 1

        stats = {
            "total_loss": total_loss / max(num_batches, 1),
            "policy_loss": total_pi_loss / max(num_batches, 1),
            "value_loss": total_v_loss / max(num_batches, 1),
            "num_examples": len(examples),
            "iteration": self.iteration,
        }
        self.train_history.append(stats)
        logger.info(f"Iteration {self.iteration}: loss={stats['total_loss']:.4f} (pi={stats['policy_loss']:.4f}, v={stats['value_loss']:.4f})")
        return stats

    def _update_lr(self):
        for milestone, lr in self.lr_schedule:
            if self.iteration >= milestone:
                for param_group in self.optimizer.param_groups:
                    param_group["lr"] = lr

    def save_checkpoint(self, filename=None) -> str:
        if filename is None:
            filename = f"checkpoint_{self.iteration:04d}.pt"
        filepath = self.checkpoint_dir / filename
        torch.save({"iteration": self.iteration, "model_state_dict": self.net.state_dict(),
                    "optimizer_state_dict": self.optimizer.state_dict(), "train_history": self.train_history}, filepath)
        return str(filepath)

    def load_checkpoint(self, filepath):
        checkpoint = torch.load(filepath, map_location=self.device)
        self.net.load_state_dict(checkpoint["model_state_dict"])
        self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        self.iteration = checkpoint["iteration"]

---
## 9. Arena & Baselines

모델 평가 프레임워크:
- **Arena**: 두 MCTS 에이전트 간 대전
- **Baselines**: RandomAgent, GreedyAgent, WeakMCTSAgent
- **Pattern Evaluation**: 패턴별 Q-side 승률 측정

Source: `handichess/track_a/arena.py`, `handichess/track_a/baseline.py`

In [ ]:
"""
Arena for model evaluation.

Plays games between two agents (or against a baseline) to measure
relative strength. Used for:
  - Deciding whether to accept a new model version
  - Evaluating per-pattern performance (Track A → gamelog)
  - Color-balanced evaluation
"""

from __future__ import annotations

import logging
from typing import Optional

import numpy as np
import chess
import chess.pgn

logger = logging.getLogger(__name__)


class Arena:
    """Plays evaluation games between two MCTS agents."""

    def __init__(self, game, net1, net2, mcts_config=None, device="cpu", seed=42):
        self.game = game
        self.net1 = net1
        self.net2 = net2
        self.device = device
        self.seed = seed
        mc = mcts_config or {}
        self.num_simulations = mc.get("num_simulations", 800)
        self.c_puct = mc.get("c_puct", 1.25)

    def play_game(self, player1_starts=True):
        """Play a single game. Returns (result_for_player1, num_moves)."""
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)
        
        mcts1 = MCTS(self.game, self.net1, num_simulations=self.num_simulations, c_puct=self.c_puct, device=self.device)
        mcts2 = MCTS(self.game, self.net2, num_simulations=self.num_simulations, c_puct=self.c_puct, device=self.device)

        state = self.game.get_init_board()
        player = 1
        move_count = 0

        while True:
            result = self.game.get_game_ended(state, player)
            if result != 0:
                break
            if move_count >= 512:
                result = 1e-4
                break

            mcts = mcts1 if (player == 1) == player1_starts else mcts2
            temp = 1.0 if move_count < 10 else 0.0
            add_n = move_count < 10
            action_probs = mcts.search(state, player, temperature=temp, add_noise=add_n)

            if temp > 0:
                action = np.random.choice(len(action_probs), p=action_probs)
            else:
                action = np.argmax(action_probs)
            state, player = self.game.get_next_state(state, player, action)
            move_count += 1

        if abs(result) < 0.01:
            return 0.0, move_count
        elif player1_starts:
            return (float(np.sign(result)), move_count) if player == 1 else (-float(np.sign(result)), move_count)
        else:
            return (-float(np.sign(result)), move_count) if player == 1 else (float(np.sign(result)), move_count)

    def play_games(self, num_games):
        """Play multiple color-balanced games."""
        wins = draws = losses = 0
        total_moves = 0

        for i in range(num_games):
            result, moves = self.play_game(i % 2 == 0)
            total_moves += moves
            if result > 0.5: wins += 1
            elif result < -0.5: losses += 1
            else: draws += 1

        total = wins + draws + losses
        return {
            "player1_wins": wins, "player1_draws": draws, "player1_losses": losses,
            "player1_win_rate": (wins + 0.5 * draws) / max(total, 1),
            "total_games": total, "avg_moves": total_moves / max(total, 1),
        }

In [ ]:
"""
Weak baseline agents for measuring training progress.
Provides random and low-simulation MCTS agents.
"""

class RandomAgent:
    """Plays uniformly random legal moves."""
    def __init__(self, game):
        self.game = game

    def get_action(self, state, player):
        valid = self.game.get_valid_moves(state, player)
        valid_actions = np.where(valid > 0)[0]
        return np.random.choice(valid_actions)


class GreedyAgent:
    """Plays the move with highest policy probability, no search."""
    def __init__(self, game, net, device="cpu"):
        self.game = game
        self.net = net
        self.device = device

    def get_action(self, state, player):
        canonical = self.game.get_canonical_form(state, player)
        encoded = self.game.get_encoded_state(canonical)
        valid = self.game.get_valid_moves(state, player)

        encoded_t = torch.FloatTensor(encoded).to(self.device)
        valid_t = torch.FloatTensor(valid).to(self.device)
        policy, _ = self.net.predict(encoded_t, valid_t)
        policy = policy.cpu().numpy() * valid
        return int(np.argmax(policy))


class WeakMCTSAgent:
    """MCTS agent with very few simulations."""
    def __init__(self, game, net, num_simulations=25, device="cpu"):
        self.game = game
        self.mcts = MCTS(game, net, num_simulations=num_simulations, device=device)

    def get_action(self, state, player):
        probs = self.mcts.search(state, player, temperature=0.0, add_noise=False)
        return int(np.argmax(probs))


def play_against_baseline(game, net, baseline_type="random", num_games=20, mcts_config=None, device="cpu"):
    """Play the trained network against a baseline agent."""
    mc = mcts_config or {}

    if baseline_type == "random":
        baseline = RandomAgent(game)
    elif baseline_type == "greedy":
        baseline = GreedyAgent(game, net, device)
    elif baseline_type == "weak_mcts":
        baseline = WeakMCTSAgent(game, net, num_simulations=25, device=device)
    else:
        raise ValueError(f"Unknown baseline: {baseline_type}")

    mcts = MCTS(game, net, num_simulations=mc.get("num_simulations", 200), c_puct=mc.get("c_puct", 1.25), device=device)

    wins = draws = losses = 0
    for i in range(num_games):
        state = game.get_init_board()
        player = 1
        trained_is_player1 = (i % 2 == 0)
        move_count = 0

        while move_count < 512:
            result = game.get_game_ended(state, player)
            if result != 0:
                break
            is_trained = (player == 1) == trained_is_player1
            if is_trained:
                probs = mcts.search(state, player, temperature=0.0, add_noise=False)
                action = int(np.argmax(probs))
            else:
                action = baseline.get_action(state, player)
            state, player = game.get_next_state(state, player, action)
            move_count += 1

        result = game.get_game_ended(state, player)
        if abs(result) < 0.01:
            draws += 1
        else:
            if trained_is_player1:
                if (player == 1 and result > 0) or (player == -1 and result < 0): wins += 1
                else: losses += 1
            else:
                if (player == -1 and result > 0) or (player == 1 and result < 0): wins += 1
                else: losses += 1

    total = wins + draws + losses
    return {"wins": wins, "draws": draws, "losses": losses,
            "win_rate": (wins + 0.5 * draws) / max(total, 1), "baseline": baseline_type}

---
## 10. Training Script (CLI Entry Point)

전체 학습 파이프라인을 실행하는 CLI 스크립트입니다.  
TicTacToe 검증 → Chess 학습의 순서로 사용합니다.

```bash
# Usage examples:
python scripts/train.py --game tictactoe --iterations 50
python scripts/train.py --game chess --iterations 100 --games-per-iter 256
```

Source: `scripts/train.py`

In [ ]:
# Training script (scripts/train.py) — CLI entry point
# This script orchestrates: Self-play → Training → Checkpoint
#
# Key arguments:
#   --game {tictactoe, chess}     Game to train on
#   --iterations N                Number of training iterations
#   --games-per-iter N            Self-play games per iteration
#   --simulations N               MCTS simulations per move
#   --pattern PATTERN_ID          Handicap pattern (chess only)
#   --device {cpu, cuda, mps}     Torch device
#   --num-workers N               Parallel worker processes
#   --devices cuda:5,cuda:7       Multi-GPU device list
#
# Training loop per iteration:
#   Phase 1: Self-play (generate games → collect training examples)
#   Phase 2: Training (sample from replay buffer → optimize network)
#   Phase 3: Checkpoint (save every 10 iterations)

print("See scripts/train.py for the full CLI training script.")
print("Example: python scripts/train.py --game chess --iterations 100 --games-per-iter 256 --device cuda")